# Export Code

In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import os
print(os.getcwd())

/home/groups/brg/nshaheed/music2latent-streaming/notebooks


In [3]:
import os 
# os.chdir("/data/nils/repos/codecs_benchmark/music2latent")
os.chdir("/home/groups/brg/nshaheed/music2latent-streaming/")

from music2latent.hparams import hparams
from music2latent import EncoderDecoder
from music2latent.transforms import StreamableSTFT

from music2latent.config_loader import load_config

from music2latent.ema import ExponentialMovingAverage
from music2latent.models_stream import *
import torch
from IPython.display import display, Audio
torch.set_grad_enabled(False)

def get_models(load_path_inference = None):
    gen = UNet()
    if load_path_inference is not None:
        checkpoint = torch.load(load_path_inference, map_location="cpu")
        gen.load_state_dict(checkpoint['gen_state_dict'], strict=False)
        # if checkpoint['ema_state_dict'] exists, init ema model and load ema_state_dict
        if 'ema_state_dict' in checkpoint:
            ema = ExponentialMovingAverage(gen.parameters(), decay=hparams.ema_momentum)
            ema.load_state_dict(checkpoint['ema_state_dict'])
            ema.copy_to()
            with ema.average_parameters():
                checkpoint['gen_state_dict'] = gen.state_dict()
        gen.load_state_dict(checkpoint['gen_state_dict'], strict=True)
        # self.gen = torch.jit.script(gen)
    return gen

/home/groups/brg/nshaheed/music2latent-streaming/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 3.2.1'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [4]:
import nn_tilde
import torch
import cached_conv as cc 

cc.MAX_BATCH_SIZE = 1

class StreamingM2L(nn_tilde.Module):
    def __init__(self, net, transform):
        super().__init__()
        self.net = net
        self.transform = transform 
        self.comp_ratio = 4096
        self.latent_size = 64
        self.sigma_rescale = 0.06
        self.diffusion_steps = 1 
        self.hop = hparams.hop
        self.sigma_min = hparams.sigma_min
        self.sigma_max = hparams.sigma_max
        self.rho = hparams.rho
        self.freq_downsample_list = hparams.freq_downsample_list
        self.mixed_precision = hparams.mixed_precision
            
        self.register_method(
                "encode",
                in_channels=1,
                in_ratio=1,
                out_channels=self.latent_size,
                out_ratio=self.comp_ratio,
                input_labels=['(signal) Audio in'],
                output_labels=[f"latent {i}" for i in range(self.latent_size)],
                test_buffer_size=32768*2,
            )

        self.register_method("decode",
                            in_channels=self.latent_size,
                            in_ratio=self.comp_ratio,
                            out_channels=1,
                            out_ratio=1,
                            test_buffer_size=32768*2,
                            input_labels=[
                                f'(signal) Latent dimension {i+1}'
                                for i in range(self.latent_size)
                            ],
                            output_labels=[
                                '(signal) Audio out'
                            ])
        
    
    
    def get_sigma(self, i: int, k: int):
        sigma_min, sigma_max = self.sigma_min, self.sigma_max
        return (sigma_min**(1./self.rho) + ((i-1)/(k-1))*(sigma_max**(1./self.rho)-sigma_min**(1./self.rho)))**self.rho

    def denoise(self, noisy_samples: torch.Tensor, sigma: float, latents: torch.Tensor):
        # Denoise samples
        # with torch.no_grad():
        #     with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=True):
        #         print(latents.shape, noisy_samples.shape, sigma)
        pred_samples = self.net.forward_generator(latents, noisy_samples, sigma)
        # Sample noise
        pred_noises = torch.randn_like(pred_samples)
        return pred_noises, pred_samples

    def reverse_step(self, x: torch.Tensor, noise: torch.Tensor, sigma: float):
        return x + ((sigma**2 - self.sigma_min**2)**0.5)*noise

    def reverse_diffusion(self, initial_noise: torch.Tensor, latents: torch.Tensor):
        next_noisy_samples = initial_noise
        # Reverse process step-by-step
        # for k in range(self.diffusion_steps):
        k = 0
        # Get sigma values
        sigma = self.get_sigma(self.diffusion_steps+1-k, self.diffusion_steps+1)
        # next_sigma = self.get_sigma(self.diffusion_steps-k, self.diffusion_steps+1)

        # Denoise 
        noisy_samples = next_noisy_samples
        _, pred_samples = self.denoise(noisy_samples, sigma, latents)

        # Step to next (lower) noise level
        # next_noisy_samples = self.reverse_step(pred_samples, pred_noises, next_sigma)

        return pred_samples
    
    
    def decode_to_representation(self, latents: torch.Tensor):
        latents = latents*self.sigma_rescale
        num_samples = latents.shape[0]
        downscaling_factor = 2**self.freq_downsample_list.count(0)
        sample_length = int(latents.shape[-1]*downscaling_factor)
        initial_noise = torch.randn((num_samples, 2, self.hop*2, sample_length))*self.sigma_max
        
        decoded_spectrograms = self.reverse_diffusion(initial_noise, latents=latents)
        return decoded_spectrograms



    @torch.jit.export
    def encode(self, x: torch.Tensor) -> torch.Tensor:
        n = x.shape[0]
        repr_encoder = self.transform.forward(x[:1])
        latent = self.net.encoder(repr_encoder, extract_features=False)/self.sigma_rescale
        return latent.repeat(n, 1, 1)
    
    
    @torch.jit.export
    def decode(self, latent: torch.Tensor) -> torch.Tensor:
        n = latent.shape[0]
        repr = self.decode_to_representation(latent[:1])
        waveform = self.transform.inverse(repr)
        return waveform.repeat(n, 1, 1)

In [5]:
cc.use_cached_conv(True)

config = "/home/groups/brg/nshaheed/music2latent-streaming/checkpoints/2025-07-24 12:15:19.566459/config.py"
load_config(config)
ckpt_path = "/home/groups/brg/nshaheed/music2latent-streaming/checkpoints/2025-07-24 12:15:19.566459/model_fid_1.3627121132819016_loss_100.68_iters_560000.pt"
gen = get_models(ckpt_path)
transform = StreamableSTFT(nfft = hparams.hop*4, hop_size = hparams.hop, skip_features = 1, stream = True)


Using conv mode:  causal


/tmp/ipykernel_174982/3412542380.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(load_path_inference, map_location="cpu")


In [6]:
model = StreamingM2L(net=gen, transform = transform)

init cache wth  torch.Size([1, 64, 1024, 128])
init cache wth  torch.Size([1, 64, 1024, 128])
init cache wth  torch.Size([1, 64, 1024, 128])
init cache wth  torch.Size([1, 64, 512, 128])
init cache wth  torch.Size([1, 128, 512, 128])
init cache wth  torch.Size([1, 128, 512, 128])
init cache wth  torch.Size([1, 128, 256, 64])
init cache wth  torch.Size([1, 256, 256, 64])
init cache wth  torch.Size([1, 256, 256, 64])
init cache wth  torch.Size([1, 256, 256, 64])
init cache wth  torch.Size([1, 256, 128, 32])
init cache wth  torch.Size([1, 256, 128, 32])
init cache wth  torch.Size([1, 256, 128, 32])
init cache wth  torch.Size([1, 256, 128, 32])
init cache wth  torch.Size([1, 256, 64, 16])
init cache wth  torch.Size([1, 256, 64, 16])
init cache wth  torch.Size([1, 256, 64, 16])
init cache wth  torch.Size([1, 256, 64, 16])
init cache wth  torch.Size([1, 512, 16])
init cache wth  torch.Size([1, 512, 16])
init cache wth  torch.Size([1, 512, 16])
init cache wth  torch.Size([1, 512, 16])
init ca

In [7]:
ts_model = model.export_to_ts("test_export.ts")

In [7]:
torch.set_grad_enabled(False)
ts_model = torch.jit.load('test_export.ts')

In [ ]:
waveform = torch.randn(1, 1, 65536)
latents = ts_model.encode(waveform)
out = ts_model.decode(latents)
display(Audio(out.squeeze(), rate = 44100))

In [ ]:
import torch
from IPython.display import display, Audio

# Parameters
num_samples = 65536
sample_rate = 44100  # You can change this as needed
frequencies = [220, 440, 880]  # Frequencies in Hz

# Time vector
t = torch.arange(num_samples) / sample_rate  # Shape: (131072,)

# Sum of sinewaves
signal = sum(torch.sin(2 * torch.pi * f * t) for f in frequencies)


display(Audio(signal.squeeze(), rate = sample_rate))

In [ ]:
waveform = signal.reshape(1, 1, -1)
latents = ts_model.encode(waveform)
out = ts_model.decode(latents)
display(Audio(out.squeeze().detach(), rate = 44100))

In [ ]:
chunk_size = 4196

allout = []
for j in range(waveform.shape[-1]//chunk_size):
    wavechunk = waveform[..., j*chunk_size: (j+1)*chunk_size]
    latents = ts_model.encode(wavechunk)
    out = ts_model.decode(latents)
    allout.append(out)
allout = torch.cat(allout, -1)
display(Audio(allout.squeeze(), rate = 44100))


### Further tests

In [ ]:
waveform = signal.reshape(1, 1, -1)

transform = StreamableSTFT(nfft = hparams.hop*4, hop_size = hparams.hop, skip_features = 1, stream = True)

chunk_size = 4096

allout = []
for j in range(waveform.shape[-1]//chunk_size):
    wavechunk = waveform[..., j*chunk_size: (j+1)*chunk_size]
    latents = model.encode(wavechunk)
    allout.append(latents)

allout = torch.cat(allout, -1)

allout = model.decode(allout)
display(Audio(allout.squeeze(), rate = 44100))

In [ ]:
chunk_size = 4196

allout = []
for j in range(waveform.shape[-1]//chunk_size):
    wavechunk = waveform[..., j*chunk_size: (j+1)*chunk_size]
    latents = model.encode(wavechunk)
    out = model.decode_to_representation(latents)
    allout.append(out)
allout = torch.cat(allout, -1)

test = model.transform.inverse(allout)
display(Audio(test.squeeze(), rate = 44100))

repr = allout

chunk_size = 1
allout = []
for j in range(repr.shape[-1]//chunk_size):
    wavechunk = repr[..., j*chunk_size: (j+1)*chunk_size]
    # latents = model.encode(wavechunk)
    out = model.transform.inverse(wavechunk)
    allout.append(out)
allout = torch.cat(allout, -1)

display(Audio(allout.squeeze(), rate = 44100))

In [ ]:
chunk_size = 2048

allout = []
for j in range(waveform.shape[-1]//chunk_size):
    wavechunk = waveform[..., j*chunk_size: (j+1)*chunk_size]
    latents = model.transform.forward(wavechunk)
    out = model.transform.inverse(latents)
    allout.append(out)
allout = torch.cat(allout, -1)
display(Audio(allout.squeeze(), rate = 44100))

In [ ]:
chunk_size = 4196

allout = []
for j in range(waveform.shape[-1]//chunk_size):
    wavechunk = waveform[..., j*chunk_size: (j+1)*chunk_size]
    latents = model.encode(wavechunk)
    out = model.decode(latents)
    allout.append(out)
allout = torch.cat(allout, -1)
display(Audio(allout.squeeze(), rate = 44100))

In [ ]:
for j in range(waveform.shape[-1]//chunk_size):
allout = model.transform.inverse(allout)
display(Audio(allout.squeeze(), rate = 44100))

In [ ]:

config = "/data/nils/repos/codecs_benchmark/music2latent/checkpoints/2025-06-19 15:44:37.552665/config_nogn.py"
load_config(config)

gen = get_models(None)

In [77]:
transform = StreamableSTFT(nfft = hparams.hop*4, hop_size = hparams.hop, skip_features = 1, stream = True)
waveform = torch.randn(1, 1, 262144)
rand=  transform.forward(waveform)

In [ ]:
def use_cached_conv(state: bool):
    global USE_BUFFER_CONV
    USE_BUFFER_CONV = state
    
def chunk_process(f, x, N):
    x = torch.split(x, x.shape[-1] // N, -1)
    y = torch.cat([f(_x) for _x in x], -1)
    return y


def test_equal(model_constructor, input_tensor, crop=True, cd=50):

    use_cached_conv(False)
    model = model_constructor()
    use_cached_conv(True)
    cmodel = model_constructor()

    for p1, p2 in zip(model.parameters(), cmodel.parameters()):
        p2.data.copy_(p1.data)

    y = model(input_tensor)#[..., :-cd]
    cy = chunk_process(lambda x: cmodel(x), input_tensor, 4)#[..., cd:]

    if crop:
        cd = cd
        y = y[..., cd:]
        cy = cy[..., cd:]
    
    print(y.shape)
    return torch.allclose(y, cy, 1e-4, 1e-4), y, cy


model_constructor = lambda  : get_models(None).encoder

input_tensor = rand
test, y, cy = test_equal(input_tensor = rand,  model_constructor =model_constructor)
print(test)

In [ ]:
print(get_models(None).encoder)

In [ ]:
print(y[0,0])
print(cy[0,0])

In [ ]:
def use_cached_conv(state: bool):
    global USE_BUFFER_CONV
    USE_BUFFER_CONV = state
    
def chunk_process(f, x, N, y):
    x = torch.split(x, x.shape[-1] // N, -1)
    y = torch.split(y, y.shape[-1] // N, -1)
    y = torch.cat([f( _y,_x, 80.) for (_x, _y) in zip(x,y)], -1)
    return y


def test_equal(model_constructor, input_tensor1, input_tensor2=None,  crop=True, cd=32):

    use_cached_conv(False)
    model = model_constructor()
    use_cached_conv(True)
    cmodel = model_constructor()

    for p1, p2 in zip(model.parameters(), cmodel.parameters()):
        p2.data.copy_(p1.data)

    y = model.forward_generator(input_tensor2, input_tensor1, 80.)#[..., :-cd]
    cy = chunk_process(lambda x, y , z: cmodel.forward_generator(x, y, z), input_tensor1, 4, input_tensor2)#[..., cd:]

    if crop:
        cd = cd
        y = y[..., cd:]
        cy = cy[..., cd:]
    
    print(y.shape)
    return torch.allclose(y, cy, 1e-4, 1e-4), y, cy


model_constructor = lambda  : get_models(None).encoder

model_constructor = lambda  : get_models(None)

input_tensor1 = torch.randn((1, 2, 1024, 128*4))
input_tensor2 = torch.randn((1, 64, 16*4))


test, y, cy = test_equal(input_tensor1 = input_tensor1,  model_constructor =model_constructor, input_tensor2 = input_tensor2)
print(test)

In [ ]:
print(y[0,0, :10])
print(cy[0,0, :10])